# DSCI 550 — Assignment 3: Decision Trees and K-Means

**Fall 2026 · 100 points · 13 questions**

This assignment focuses on three core topics:

1. decision-tree classification;
2. decision-tree regression;
3. K-means clustering.

Classification and regression are supervised-learning tasks. K-means clustering is an unsupervised-learning task.

## Data

The provided setup cell creates three clean numerical datasets, one for each part of the assignment.

The datasets contain no missing values. **Data cleaning is not part of this assignment.**

## Required libraries and fixed settings

You must use the following scikit-learn tools and settings exactly.

| Task | Required tool | Fixed settings |
|---|---|---|
| Classification tree | `sklearn.tree.DecisionTreeClassifier` | entropy criterion, maximum depth 3, random state 42 |
| Regression tree | `sklearn.tree.DecisionTreeRegressor` | maximum depth 3, random state 42 |
| Standardization | `sklearn.preprocessing.StandardScaler` | default settings |
| K-means | `sklearn.cluster.KMeans` | settings are stated in each K-means question |
| Classification metric | `sklearn.metrics.accuracy_score` | default settings |
| Regression metric | `sklearn.metrics.mean_squared_error` | |

For calculations that the question asks you to perform manually, use NumPy/Pandas rather than replacing the calculation with another machine-learning helper.

## Instructions

- Run the provided setup cell first.
- Complete each question in its own answer cell.
- Store the requested result in the exact variable name `answer_1`, ..., `answer_13`.
- Round numerical answers to six decimal places only when the question says to round.
- Use the required library/class and fixed hyperparameters stated for the question.
- Do not change the provided datasets, train/test splits, centroids, seeds, or elbow function.
- Questions are graded independently. Do not make one answer cell depend on an earlier `answer_*` variable.
- There are no notebook-side checker cells.
- Before submitting, restart the runtime, run all cells top-to-bottom, and save the notebook with outputs.

## Point allocation

| Part | Topic | Points |
|---|---|---:|
| 1 | Decision-Tree Classification | 35 |
| 2 | Decision-Tree Regression | 25 |
| 3 | K-Means Clustering | 40 |
| **Total** |  | **100** |

In [ ]:
# PROVIDED SETUP — run first. Do not modify.

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification, make_regression, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, mean_squared_error

RANDOM_STATE = 42

# ---------------------------------------------------------------------
# Classification dataset
# ---------------------------------------------------------------------
_X_class, _y_class = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    weights=[0.45, 0.55],
    class_sep=1.15,
    flip_y=0.02,
    random_state=RANDOM_STATE,
)

_X_class = pd.DataFrame(
    _X_class,
    columns=[f"x{i}" for i in range(_X_class.shape[1])]
)
_y_class = pd.Series(_y_class, name="target")

X_CLASS_TRAIN, X_CLASS_TEST, Y_CLASS_TRAIN, Y_CLASS_TEST = train_test_split(
    _X_class,
    _y_class,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=_y_class,
)

# ---------------------------------------------------------------------
# Regression dataset
# ---------------------------------------------------------------------
_X_reg, _y_reg = make_regression(
    n_samples=4000,
    n_features=10,
    n_informative=6,
    noise=18.0,
    random_state=RANDOM_STATE,
)

_X_reg = pd.DataFrame(
    _X_reg,
    columns=[f"x{i}" for i in range(_X_reg.shape[1])]
)
_y_reg = pd.Series(_y_reg, name="target")

X_REG_TRAIN, X_REG_TEST, Y_REG_TRAIN, Y_REG_TEST = train_test_split(
    _X_reg,
    _y_reg,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

# ---------------------------------------------------------------------
# Clustering dataset
# ---------------------------------------------------------------------
_X_cluster, _unused_cluster_labels = make_blobs(
    n_samples=1500,
    centers=[(-5.0, -3.0), (0.0, 5.0), (5.5, -1.0)],
    cluster_std=[1.15, 1.35, 0.95],
    random_state=RANDOM_STATE,
)

X_CLUSTER_RAW = pd.DataFrame(_X_cluster, columns=["x1", "x2"])

# These centroids are ALREADY in standardized feature space.
KMEANS_INITIAL_CENTROIDS_SCALED = np.array([
    [-1.05, -0.75],
    [-0.10,  1.25],
    [ 1.15, -0.45],
], dtype=float)

KMEANS_INITIALIZATION_SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

def detect_elbow(k_values, inertias):
    '''
    Supplied deterministic elbow rule.
    Returns the k with the greatest perpendicular distance from the
    line joining the first and last normalized points.
    Ties choose the smaller k.
    '''
    x = np.asarray(k_values, dtype=float)
    y = np.asarray(inertias, dtype=float)

    if x.ndim != 1 or y.ndim != 1 or len(x) != len(y) or len(x) < 3:
        raise ValueError("k_values and inertias must be 1-D arrays of equal length >= 3.")

    x_norm = (x - x.min()) / (x.max() - x.min())

    if np.isclose(y.max(), y.min()):
        return int(np.min(x))

    y_norm = (y - y.min()) / (y.max() - y.min())

    p1 = np.array([x_norm[0], y_norm[0]])
    p2 = np.array([x_norm[-1], y_norm[-1]])
    line = p2 - p1
    line_length = np.linalg.norm(line)

    distances = []
    for xi, yi in zip(x_norm, y_norm):
        offset = np.array([xi, yi]) - p1
        distance = abs(
            line[0] * offset[1] - line[1] * offset[0]
        ) / line_length
        distances.append(distance)

    distances = np.asarray(distances)
    tied = np.flatnonzero(np.isclose(distances, distances.max()))
    return int(x[tied[0]])

print("Classification:", X_CLASS_TRAIN.shape, "train,", X_CLASS_TEST.shape, "test")
print("Regression:", X_REG_TRAIN.shape, "train,", X_REG_TEST.shape, "test")
print("Clustering:", X_CLUSTER_RAW.shape)

## Part 1 — Decision-Tree Classification (35 points)

Use `X_CLASS_TRAIN`, `Y_CLASS_TRAIN`, `X_CLASS_TEST`, and `Y_CLASS_TEST`.

For every classification-tree question, use `sklearn.tree.DecisionTreeClassifier` with:

- entropy as the split criterion;
- maximum depth 3;
- random state 42.

### Q1 — Train the classification tree (6 pts)

Train the required classification tree on the supplied classification training set.

Store the **number of nodes in the fitted tree** as an integer in `answer_1`.

Use the tree structure exposed by the fitted scikit-learn model; do not estimate the node count manually.

In [ ]:
# YOUR CODE HERE
answer_1 = None

### Q2 — Entropy of the training target (8 pts)

Using `Y_CLASS_TRAIN`, calculate the target entropy in **bits** using the definition introduced in lecture.

For this question, compute the class probabilities and entropy yourself with NumPy/Pandas. Do not obtain the value from a fitted tree and do not use an entropy helper from another library.

Store the result, rounded to six decimal places, in `answer_2`.

In [ ]:
# YOUR CODE HERE
answer_2 = None

### Q3 — Information gain at the root (12 pts)

Fit the required classification tree on the supplied classification training set.

Inspect the fitted model to obtain the **feature and threshold used at the root**. Apply that root split to the training observations, determine the class counts on the two sides, and calculate the information gain using the definition introduced in lecture.

Do not use `feature_importances_` as a substitute.

Store the result, rounded to six decimal places, in `answer_3`.

In [ ]:
# YOUR CODE HERE
answer_3 = None

### Q4 — Classification test performance (9 pts)

Fit the required classification tree on the supplied classification training set.

Use `sklearn.metrics.accuracy_score` to evaluate its predictions on `X_CLASS_TEST` against `Y_CLASS_TEST`.

Store the test accuracy, rounded to six decimal places, in `answer_4`.

In [ ]:
# YOUR CODE HERE
answer_4 = None

## Part 2 — Decision-Tree Regression (25 points)

Use `X_REG_TRAIN`, `Y_REG_TRAIN`, `X_REG_TEST`, and `Y_REG_TEST`.

For every regression-tree question, use `sklearn.tree.DecisionTreeRegressor` with:

- maximum depth 3;
- random state 42.

### Q5 — Train the regression tree (6 pts)

Train the required regression tree on the supplied regression training set.

Store the **number of terminal leaves** as an integer in `answer_5`.

Use the fitted scikit-learn tree model to obtain the leaf count.

In [ ]:
# YOUR CODE HERE
answer_5 = None

### Q6 — Locate the first test observation's leaf (5 pts)

Fit the required regression tree for this question.

Use the fitted model's leaf-assignment functionality to determine which terminal leaf receives the **first row of `X_REG_TEST`**.

Store that leaf/node ID as an integer in `answer_6`.

In [ ]:
# YOUR CODE HERE
answer_6 = None

### Q7 — Verify how a regression leaf predicts (7 pts)

Fit the required regression tree.

Determine the terminal leaf reached by the first row of `X_REG_TEST`. Then determine which rows of `X_REG_TRAIN` reach that same leaf.

Compute the mean of the corresponding values in `Y_REG_TRAIN`.

Store the mean, rounded to six decimal places, in `answer_7`.

This verifies the regression-tree leaf prediction described in lecture.

In [ ]:
# YOUR CODE HERE
answer_7 = None

### Q8 — Regression test performance (7 pts)

Fit the required regression tree on the supplied regression training set.

Predict the targets for `X_REG_TEST`. Use `sklearn.metrics.mean_squared_error` and compute the **root-mean-square error (RMSE)**.

Store the RMSE, rounded to six decimal places, in `answer_8`.

In [ ]:
# YOUR CODE HERE
answer_8 = None

## Part 3 — K-Means Clustering (40 points)

Use `X_CLUSTER_RAW`.

For standardization, use `sklearn.preprocessing.StandardScaler`.

When a question asks you to fit K-means, use `sklearn.cluster.KMeans` with the exact settings stated in that question.

The supplied variable `KMEANS_INITIAL_CENTROIDS_SCALED` contains centroids that are already expressed in standardized feature space.

### Q9 — Standardize the clustering data (7 pts)

Fit `sklearn.preprocessing.StandardScaler` to the full `X_CLUSTER_RAW` dataset and transform it.

Store the **first transformed observation** as a tuple of two Python floats, each rounded to six decimal places, in `answer_9`.

In [ ]:
# YOUR CODE HERE
answer_9 = None

### Q10 — Perform one K-means assignment manually (8 pts)

Standardize `X_CLUSTER_RAW` using `StandardScaler`.

Using the first standardized observation and `KMEANS_INITIAL_CENTROIDS_SCALED`, compute the Euclidean distance to each supplied centroid with NumPy.

Store the zero-based index of the nearest centroid as an integer in `answer_10`.

Do not use `KMeans.fit`, `KMeans.predict`, or another clustering routine for this question.

In [ ]:
# YOUR CODE HERE
answer_10 = None

### Q11 — Perform one centroid update manually (8 pts)

Standardize `X_CLUSTER_RAW` using `StandardScaler`.

Using NumPy and `KMEANS_INITIAL_CENTROIDS_SCALED`:

1. assign every standardized observation to its nearest supplied centroid using Euclidean distance;
2. select the observations assigned to centroid 0;
3. compute their coordinate-wise mean.

Store the updated centroid-0 coordinates as a tuple of two Python floats rounded to six decimal places in `answer_11`.

Do not fit a `KMeans` model for this question.

In [ ]:
# YOUR CODE HERE
answer_11 = None

### Q12 — Compare K-means initialization methods (8 pts)

Standardize `X_CLUSTER_RAW` using `StandardScaler`.

For every value in `KMEANS_INITIALIZATION_SEEDS`, fit two `sklearn.cluster.KMeans` models with:

- 3 clusters;
- exactly one initialization run (`n_init=1`);
- the current supplied seed as `random_state`.

Use `"random"` initialization for one model and `"k-means++"` for the other.

Compute the mean final inertia across all supplied seeds for each initialization method.

Store exactly `"random"` or `"k-means++"` in `answer_12`, choosing the method with the lower mean inertia. If the means tie, choose `"k-means++"`.

In [ ]:
# YOUR CODE HERE
answer_12 = None

### Q13 — Choose the number of clusters with the supplied elbow rule (9 pts)

Standardize `X_CLUSTER_RAW` using `StandardScaler`.

For each `k` from 1 through 8, fit `sklearn.cluster.KMeans` with:

- `k` clusters;
- K-means++ initialization;
- 20 initialization runs (`n_init=20`);
- random state 42.

Record the final inertia for each `k`, preserving the order from 1 through 8.

Pass those `k` values and inertia values to the supplied `detect_elbow` function.

Store the integer returned by `detect_elbow` in `answer_13`.

In [ ]:
# YOUR CODE HERE
answer_13 = None

---

## Submission checklist

Before submitting:

- restart the kernel/runtime;
- run the notebook from top to bottom;
- confirm `answer_1` through `answer_13` are defined;
- save the `.ipynb` with outputs.
